# AI-Powered Resume Screening Assistant - Colab Runner

This notebook runs the thesis project end to end in Google Colab.

## Project Purpose

The project builds an explainable AI assistant for resume screening. It helps compare candidate resumes against a job description, rank candidates, and show evidence such as matched skills, missing skills, experience signals, and education/certification signals.

The system is designed as a decision-support tool for HR screening. It does not make final hiring decisions.

## Main Research Goals

1. Compare classical information retrieval models such as TF-IDF and BM25 with semantic embedding models.
2. Combine semantic similarity with interpretable evidence using a hybrid score.
3. Store resume and job description chunks in Qdrant as a vector database.
4. Prepare retrieved chunks as grounded context for future RAG explanations and HR question answering.
5. Evaluate ranking quality with reproducible metrics when labeled resume-job pairs are available.


## How to Use This Notebook

Recommended Colab setup:

1. Upload the full `ai_resume_screening` project folder to Colab or Google Drive.
2. Open this notebook in Colab.
3. Run cells from top to bottom.
4. Start with TF-IDF or BM25 because they are fast.
5. Run the Qdrant section when you want to create and query the vector database.

Runtime recommendation: CPU is enough for TF-IDF, BM25, and small Qdrant demos. GPU is useful for larger embedding models.


In [ ]:
# Optional: mount Google Drive if the project is stored there.
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os
import sys

candidate_dirs = [
    Path.cwd(),
    Path.cwd().parent,
    Path('/content/ai_resume_screening'),
    Path('/content/thesis/ai_resume_screening'),
    Path('/content/drive/MyDrive/ai_resume_screening'),
    Path('/content/drive/MyDrive/thesis/ai_resume_screening'),
]

PROJECT_DIR = None
for path in candidate_dirs:
    if (path / 'requirements.txt').exists() and (path / 'src').exists():
        PROJECT_DIR = path.resolve()
        break

if PROJECT_DIR is None:
    raise FileNotFoundError(
        'Could not find the ai_resume_screening project folder. Upload the project to /content/ai_resume_screening '
        'or mount Google Drive and update candidate_dirs in this cell.'
    )

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print('Project directory:', PROJECT_DIR)
print('Files:', sorted([p.name for p in PROJECT_DIR.iterdir()])[:20])


## Install Dependencies

This installs the project dependencies in the Colab runtime. The first run can take several minutes.


In [ ]:
%pip install -q -r requirements.txt


## System Architecture

The project has two related paths:

1. Ranking path: parse documents, clean text, score resumes against the job description, compute hybrid score, and rank candidates.
2. RAG preparation path: chunk documents, embed chunks, store vectors in Qdrant, and retrieve relevant chunks for grounded explanation prompts.

Qdrant is not the final ranking model. It is used as a vector database for retrieval evidence.

```text
Resume/JD files
  -> Document parsing
  -> Text cleaning and chunking
  -> Matching models: TF-IDF, BM25, SBERT, E5, BGE, Google Embeddings
  -> Hybrid score: semantic + skills + experience + education
  -> Ranked candidates and explanations

Cleaned chunks
  -> Embedding model
  -> Qdrant vector database
  -> Top-k retrieval
  -> RAG context for LLM explanation or HR question answering
```


## Load Sample Job Description and Resumes

The default demo uses one machine learning engineer job description and three sample resumes.


In [ ]:
from config.settings import SAMPLE_JD_DIR, SAMPLE_RESUME_DIR
from src.preprocessing.document_parser import DocumentParser
from src.services.matching_service import ResumeDocument

parser = DocumentParser()
jd_files = sorted(SAMPLE_JD_DIR.glob('*.txt'))
resume_files = sorted(SAMPLE_RESUME_DIR.glob('*.txt'))

if not jd_files or not resume_files:
    raise FileNotFoundError('Sample files are missing. Check data/sample_job_descriptions and data/sample_resumes.')

jd_text = parser.parse_path(jd_files[0]).cleaned_text
resume_docs = [
    ResumeDocument(candidate_id=path.stem, filename=path.name, text=parser.parse_path(path).cleaned_text)
    for path in resume_files
]

print('Job description:', jd_files[0].name)
print('Resumes:', [doc.filename for doc in resume_docs])
print('\nJD preview:\n', jd_text[:700])


## Run Resume Matching

Use `tfidf` or `bm25` first for a quick run. Semantic models such as `sbert`, `e5`, and `bge` may download model weights on first use.


In [ ]:
import pandas as pd
from src.services.matching_service import match_resumes

model_key = 'tfidf'  # Try: 'tfidf', 'bm25', 'sbert', 'e5', 'bge'
results = match_resumes(jd_text=jd_text, resumes=resume_docs, model_key=model_key, use_hybrid_score=True)

ranking_table = pd.DataFrame([result.to_export_dict() for result in results])
display_columns = [
    'rank',
    'filename',
    'model_name',
    'final_score',
    'semantic_score',
    'recommendation',
    'matched_skills',
    'missing_skills',
]
ranking_table[display_columns]


## Compare Fast Baseline Models

This cell compares TF-IDF and BM25 on the sample data. These models are useful baselines for the thesis because they are transparent and fast.


In [ ]:
comparison_rows = []
for key in ['tfidf', 'bm25']:
    model_results = match_resumes(jd_text=jd_text, resumes=resume_docs, model_key=key, use_hybrid_score=True)
    for result in model_results:
        comparison_rows.append(
            {
                'model': key,
                'rank': result.rank,
                'candidate': result.filename,
                'final_score': result.final_score,
                'semantic_or_model_score': result.semantic_score,
                'recommendation': result.recommendation,
            }
        )

pd.DataFrame(comparison_rows).sort_values(['model', 'rank'])


## Generate an Evidence-Based Explanation

This deterministic explanation does not require an API key. If a Gemini API key is configured in `.env`, the project can also call the LLM explanation chain.


In [ ]:
from src.llm.explanation_chain import generate_candidate_explanation

top_candidate = results[0]
explanation = generate_candidate_explanation(
    result=top_candidate,
    jd_text=jd_text,
    resume_text=top_candidate.cleaned_resume_text,
    use_llm=False,
)
print(explanation)


## Qdrant Vector Database for RAG Preparation

This section embeds the job description and resumes, stores chunk vectors in Qdrant, and retrieves relevant chunks for a query.

In Colab, this notebook uses Qdrant local embedded mode through `qdrant-client`, so no Docker server is required.

The embedded dataset in this demo is:

- `data/sample_job_descriptions/machine_learning_engineer.txt`
- `data/sample_resumes/candidate_alex_python_ml.txt`
- `data/sample_resumes/candidate_brianna_frontend.txt`
- `data/sample_resumes/candidate_chris_data_analyst.txt`


In [ ]:
from src.services.vector_store_service import (
    LocalEmbeddingEncoder,
    QdrantRAGStore,
    build_rag_context,
    build_rag_documents,
)

# all-MiniLM is small and fast for Colab demos. For thesis experiments, E5 or BGE can also be used.
rag_encoder = LocalEmbeddingEncoder(model_name='sentence-transformers/all-MiniLM-L6-v2')
rag_store = QdrantRAGStore(collection_name='resume_rag_chunks_colab', encoder=rag_encoder)

rag_documents = build_rag_documents(jd_text, resume_docs)
indexed_count = rag_store.index_documents(rag_documents)
print(f'Indexed {indexed_count} chunks into Qdrant collection resume_rag_chunks_colab')


In [ ]:
query = 'Which candidates have Python, NLP, machine learning, Docker, and cloud experience?'
retrieved_chunks = rag_store.search(query=query, limit=5, document_type='resume')

retrieval_table = pd.DataFrame(
    [
        {
            'score': round(item.score, 4),
            'document_type': item.document_type,
            'filename': item.filename,
            'chunk_id': item.chunk_id,
            'preview': item.text[:250],
        }
        for item in retrieved_chunks
    ]
)
retrieval_table


In [ ]:
rag_context = build_rag_context(retrieved_chunks, max_chars=3500)
print(rag_context)


## RAG Prompt Template

The retrieved Qdrant chunks can be inserted into a prompt. This keeps the LLM grounded in evidence from the uploaded resumes and job description.


In [ ]:
rag_prompt = f'''
You are an HR screening assistant.
Use only the retrieved evidence below.
Do not infer protected attributes or make final hiring decisions.

Question:
{query}

Retrieved evidence from Qdrant:
{rag_context}

Answer format:
1. Short answer
2. Evidence by candidate
3. Missing or weak evidence
4. Suggested HR follow-up questions
'''.strip()

print(rag_prompt)


## Optional Evaluation Pipeline

If `data/processed/test_pairs.csv` exists, this cell runs the benchmark pipeline. If it does not exist, prepare the dataset pairs first using the project script.


In [ ]:
from config.settings import PROCESSED_DATA_DIR
from src.evaluation.benchmark import run_benchmark

pairs_path = PROCESSED_DATA_DIR / 'test_pairs.csv'
if pairs_path.exists():
    benchmark = run_benchmark(pairs_path=pairs_path, model_keys=['tfidf', 'bm25'], limit_jobs=20)
    display(benchmark)
else:
    print('No processed test pairs found at:', pairs_path)
    print('Prepare pairs with:')
    print('python -m src.data.prepare_pairs --primary-pairs data/raw/<ranking_dataset_file>.csv')


## Optional Streamlit Demo in Colab

Colab is mainly for experiments and notebooks, but you can expose the Streamlit UI with a tunnel. This is optional and may require extra setup.

Example using localtunnel:

```python
# !npm install -g localtunnel
# !streamlit run app.py --server.port 8501 & npx localtunnel --port 8501
```


## Thesis Summary

This project implements an explainable resume-job matching assistant. The system parses resumes and job descriptions, cleans the text, compares candidates using baseline IR models and embedding models, and computes an interpretable hybrid score. The hybrid score combines semantic similarity, required skill overlap, experience evidence, and education/certification evidence.

Qdrant improves the thesis by adding a vector database layer. Instead of only calculating similarity at runtime, the system can persist embeddings for job descriptions and resumes as searchable chunks. This makes the system RAG-ready: when HR asks a question, the system retrieves the most relevant resume/JD chunks from Qdrant and uses them as grounded context for an LLM explanation.

The final design separates scoring from generation. Ranking scores are produced by deterministic NLP models, while the LLM is constrained to explain or summarize evidence that was already retrieved or computed. This supports transparency, reproducibility, and safer use in HR screening.
